# Logging Like a Production Service

## What you'll practice

`print` is for notebooks; services and jobs use `logging`: named loggers per module, handlers, formatters, levels, structured (JSON) logs for
CloudWatch/ELK, context fields like `run_id`. You'll also see why calling `logging.basicConfig` inside libraries is a bad idea.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/files/test_06_04_logging.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/06_files_json_csv/04_logging.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · Idempotent logger setup

**Implement:** `get_logger(name: str, level: str = 'INFO', stream=None) -> logging.Logger`

Return a logger with exactly **one** `StreamHandler` (to `stream`, default `sys.stderr`) using format
`"%(asctime)s %(levelname)s %(name)s - %(message)s"`, level set from the string, and `propagate=False`. Calling it again with the same name must not add a
second handler (notebooks re-run cells!) but should update the level.

**Input:** `name: str, level: str = 'INFO', stream=None`

**Expected output:** `logging.Logger`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_get_logger_idempotent` in `tests/files/test_06_04_logging.py`.

**Check:** `pytest tests/files/test_06_04_logging.py -k test_get_logger_idempotent`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Rotating file logs

**Implement:** `file_logger(name: str, log_dir, max_bytes: int = 1_000_000, backups: int = 3) -> logging.Logger`

Return a logger writing to `log_dir/<name>.log` via `RotatingFileHandler` (create the dir). Same idempotency rule (one file handler per logger). Level DEBUG.

**Input:** `name: str, log_dir, max_bytes: int = 1_000_000, backups: int = 3`

**Expected output:** `logging.Logger`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_file_logger_rotation` in `tests/files/test_06_04_logging.py`.

**Check:** `pytest tests/files/test_06_04_logging.py -k test_file_logger_rotation`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Structured JSON logs

**Implement:** `class JsonFormatter(logging.Formatter)`

Emit one JSON object per record: `{"ts": ISO-8601 UTC with "Z", "level", "logger", "message", ...extra}` where *extra* are any attributes passed with
`logger.info("...", extra={"run_id": "r1", "rows": 10})` (i.e. record attributes not present on a plain `LogRecord`). If `exc_info` is set add `"exc"` with the
formatted traceback. Non-JSON-serializable extras → `str()`.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_json_formatter` in `tests/files/test_06_04_logging.py`.

**Check:** `pytest tests/files/test_06_04_logging.py -k test_json_formatter`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Context fields for every log line

**Implement:** `log_context(logger: logging.Logger, **fields)`

A context manager that adds `fields` (e.g. `run_id`, `file`) as attributes on **every record logged through `logger`** inside the block, using a
`logging.Filter` attached to the logger and removed on exit (even on error). Nested contexts merge (inner wins on conflicts) and restore correctly.
Existing attributes set via `extra=` on a specific call take precedence over context fields.

**Input:** `logger: logging.Logger, **fields`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_log_context, test_log_context_cleanup_on_error` in `tests/files/test_06_04_logging.py`.

**Check:** `pytest tests/files/test_06_04_logging.py -k "test_log_context or test_log_context_cleanup_on_error"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Audited job runner with structured logs · 🧩 Mini-project

**Implement:** `run_steps(steps: list[tuple[str, callable]], logger, run_id: str, clock=time.perf_counter) -> dict`

Run named steps in order. Each step is called as `step(context)` where `context` is a dict shared across steps (so step 2 can read what step 1 put there);
a step may return an int (rows processed) or `None`. For each step log `"step started"` and `"step finished"` (INFO) with extras `run_id`, `step`, `duration_ms`
(rounded int), `rows`. On exception: log `"step failed"` at ERROR with `exc_info`, mark remaining steps `"skipped"`, and stop (don't raise).
Return `{"run_id", "status": "SUCCESS"|"FAILED", "steps": [{"step", "status", "duration_ms", "rows", "error"}...], "total_rows"}` (error is `"Type: msg"` or `None`).

**Input:** `steps: list[tuple[str, callable]], logger, run_id: str, clock=time.perf_counter`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_run_steps_success, test_run_steps_failure` in `tests/files/test_06_04_logging.py`.

**Check:** `pytest tests/files/test_06_04_logging.py -k "test_run_steps_success or test_run_steps_failure"`

**YOUR CODE ↓**